# ZaureLink LiteRT-LM Export: Gemma 4 E2B → On-Device Android

**Environment:** Kaggle (TPU v5e-8, ~330GB system RAM)  
**Source Model:** QLoRA-merged `gemma-4-E2B-it` weights (from [notebook 01](./01_fine_tuning.ipynb))  
**Target Runtime:** Google LiteRT-LM (`.litertlm`)  
**Quantization:** `dynamic_wi4_afp32` (INT4 weights, FP32 activations)

### Why Kaggle TPU?

The `litert-torch export_hf` pipeline requires holding the merged FP32/16 checkpoint (~9.5GB safetensors) and quantization buffers concurrently in system RAM. Google Colab's free tier (~13GB RAM) and standard Kaggle GPU instances (~30GB RAM) both OOM'd during the export phase. The TPU v5e-8 was selected **strictly for its ~330GB system RAM headroom** — no TPU-specific computation is used.

See [notebook 01, sections 6–9](./01_fine_tuning.ipynb) for the full Colab failure log that motivated this migration.

## 1. Environment & Input Verification

Inspects the Kaggle runtime environment and verifies that the merged model artifacts (uploaded as a Kaggle Dataset from Google Drive) are present:
- 5 safetensors shards (`model-00001` through `model-00005`)
- Model configuration files (`config.json`, `generation_config.json`, `processor_config.json`)
- Tokenizer files (`tokenizer.json`, `tokenizer_config.json`, `chat_template.jinja`)

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/saxrael/zaurelink/zaurelink_merged_safe/model.safetensors.index.json
/kaggle/input/datasets/saxrael/zaurelink/zaurelink_merged_safe/config.json
/kaggle/input/datasets/saxrael/zaurelink/zaurelink_merged_safe/model-00005-of-00005.safetensors
/kaggle/input/datasets/saxrael/zaurelink/zaurelink_merged_safe/model-00001-of-00005.safetensors
/kaggle/input/datasets/saxrael/zaurelink/zaurelink_merged_safe/model-00002-of-00005.safetensors
/kaggle/input/datasets/saxrael/zaurelink/zaurelink_merged_safe/tokenizer.json
/kaggle/input/datasets/saxrael/zaurelink/zaurelink_merged_safe/tokenizer_config.json
/kaggle/input/datasets/saxrael/zaurelink/zaurelink_merged_safe/chat_template.jinja
/kaggle/input/datasets/saxrael/zaurelink/zaurelink_merged_safe/model-00004-of-00005.safetensors
/kaggle/input/datasets/saxrael/zaurelink/zaurelink_merged_safe/model-00003-of-00005.safetensors
/kaggle/input/datasets/saxrael/zaurelink/zaurelink_merged_safe/processor_config.json
/kaggle/input/datasets

## 2. Export Pipeline

The export pipeline runs in 6 stages:

1. **Workspace Preparation** — Clears previous output; creates a RAM-disk directory (`/dev/shm/`) for fast I/O
2. **Package Installation** — Installs `litert-torch-nightly`, `transformers`, and `torch`
3. **Model Export** — Runs `litert-torch export_hf` with:
   - `--quantize=dynamic_wi4_afp32` — 4-bit dynamic weight quantization, FP32 activations
   - `--cache_length=1536` — KV cache sized for 8–10 conversation turns
   - `--prefill_lengths=128,256` — Prompt prefill bucket sizes
   - `--externalize_embedder` — Separates embedder for memory optimization
   - `--bundle_litert_lm` — Packages all components into a single `.litertlm` file
   - `--use_jinja_template` — Enables custom chat template support
4. **TFLite Conversion** — PyTorch → FX Graph → MLIR → TFLite (prefill_128, prefill_256, decode sub-graphs)
5. **Dynamic Quantization** — Compresses model weights from FP32 to INT4
6. **Artifact Packaging** — Bundles model, embedder, tokenizer into final `.litertlm` container

The exported artifact is copied from RAM-disk to `/kaggle/working/` and its SHA-256 checksum is computed for integrity verification.

In [1]:
!rm -rf /kaggle/working/*
!mkdir -p /dev/shm/zaurelink_output

!pip install --upgrade torch torchvision torchaudio transformers litert-torch-nightly protobuf --break-system-packages

!export CUDA_VISIBLE_DEVICES="" && \
 litert-torch export_hf \
  --model=/kaggle/input/datasets/saxrael/zaurelink/zaurelink_merged_safe \
  --output_dir=/dev/shm/zaurelink_output \
  --quantization_recipe=dynamic_wi4_afp32 \
  --cache_length=1536 \
  --prefill_lengths=128,256 \
  --externalize_embedder \
  --bundle_litert_lm \
  --use_jinja_template

!cp /dev/shm/zaurelink_output/*.litertlm /kaggle/working/zaurelink-translator-v1.litertlm

import hashlib
from IPython.display import FileLink, display

file_path = "/kaggle/working/zaurelink-translator-v1.litertlm"
sha256_hash = hashlib.sha256()

with open(file_path, "rb") as f:
    for byte_block in iter(lambda: f.read(4096), b""):
        sha256_hash.update(byte_block)

print("\n" + "="*60)
print(f"CHECKSUM: {sha256_hash.hexdigest()}")
print("="*60 + "\n")
display(FileLink("zaurelink-translator-v1.litertlm"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 15.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 20.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 29.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 21.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 42.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 26.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 17.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 2.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 27.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 6.8 MB/s eta 0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 29.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━

/kaggle/working/zaurelink-translator-v1.litertlm

## Results & Quantization Metrics

| Component | Original Size | Quantized Size | Compression |
|---|---|---|---|
| Main Model (`model.tflite`) | 8.50 GiB | 1.09 GiB | 7.8× |
| Embedder (`embedder.tflite`) | 1.50 GiB | 198 MiB | 7.8× |
| Per-Layer Embedder (`per_layer_embedder.tflite`) | 8.75 GiB | 1.10 GiB | 8.0× |

- **Total Export Time:** 7 minutes 8 seconds
- **Final Artifact:** `zaurelink-translator-v1.litertlm`
- **SHA-256:** `48ee9a559e748dc08b9938733b53fd09d75f02d4c65114ba4b9f24795d1013bd`
- **Hosted at:** [Hugging Face — israel-ayeni/ZaureLink](https://huggingface.co/israel-ayeni/ZaureLink)

This artifact is downloaded post-install by the ZaureLink Android app via `modelManager.ts`, with resumable HTTP transfers and SHA-256 verification.